# Sparkle V2 — Fase 3.6b: agregación temporal con tolerancia a huecos

**Protocolo:** `Protocolo_Fase36_Camino_al_70.md` §3 + **Enmiendas 1 y 2** (2026-07-29).

## Qué cambia respecto del experimento 1 (§9 del registro)

El experimento 1 exigía **corridas sin huecos** de ClipIDs y obtuvo 42,9 % de cobertura. El
diagnóstico estructural mostró que la numeración de DAiSEE sí es contigua y que los huecos son
clips ausentes del conjunto de etiquetas — no saltos temporales. En el conjunto completo, sin
filtrado propio, solo el 48,5 % de los clips cae en corridas de 5 o más.

**Regla congelada (Enmienda 2):** ventana = **5 clips disponibles del mismo sujeto dentro de un
span de a lo sumo 7 posiciones de sufijo** (hasta 2 faltantes). El 7 se deriva de
`MAX_SIN_ROSTRO = 0.30` de `detector.js`: 2/7 = 28,6 % de período no observado.

**Se conserva de §9:** k=5 · etiqueta por mayoría · media de probabilidades · umbral 0,50 ·
trivial recalculado por ventana.

**Criterio (Enmienda 1):** accuracy ≥ 0,70 con cobertura de abstención ≥ 0,60 **y** límite
inferior del IC de balanced accuracy ≥ 0,5896.

**El Test no se toca.** Todo corre sobre las predicciones OOF ya guardadas.

In [ ]:
import sys, os, json, collections
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/SparkleV2'
else:
    PROJECT_DIR = os.environ.get('SPARKLE_PROJECT_DIR', './SparkleV2')

OOF_NPZ = f'{PROJECT_DIR}/fase3_5_proyeccion/oof_predicciones.npz'
OUT_36  = f'{PROJECT_DIR}/fase3_6_camino70'
os.makedirs(OUT_36, exist_ok=True)

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
import matplotlib.pyplot as plt

# ------- CONGELADOS -------
K            = 5
SPAN_MAX     = 7        # Enmienda 2: derivado de MAX_SIN_ROSTRO = 0.30
UMBRAL       = 0.50
COBERTURAS   = np.round(np.arange(1.00, 0.499, -0.05), 2)
ACC_OBJETIVO = 0.70
COB_MIN      = 0.60
BALACC_PISO  = 0.5896   # Enmienda 1: límite inferior del IC actual
N_BOOT       = 5000
SEED         = 42
BRAZO        = 'R1_d16'

z = np.load(OOF_NPZ, allow_pickle=True)
y_c    = z['y']
part_c = np.array([str(p) for p in z['part']])
ids_c  = np.array([str(c) for c in z['ids']])
p_c    = z[BRAZO]
p_R0   = z['R0'] if 'R0' in z.files else None
print(f'OOF por clip: {len(y_c)} clips · {len(set(part_c))} participantes · brazo {BRAZO}')

## 1. Construcción de ventanas con tolerancia a huecos

Celda **estructural**: opera solo sobre ClipIDs. No mira etiquetas ni predicciones.

In [ ]:
def parse(cid):
    c = os.path.splitext(str(cid).strip())[0]
    return c[:6], int(c[6:])

def ventanas_con_huecos(ids, k=K, span_max=SPAN_MAX):
    '''Ventanas de k clips disponibles del mismo sujeto dentro de span_max posiciones.
    Sin solapamiento, barrido codicioso. Devuelve listas de índices sobre `ids`.'''
    por_suj = collections.defaultdict(list)
    for i, cid in enumerate(ids):
        s, t = parse(cid); por_suj[s].append((t, i))
    wins = []
    for s, pares in por_suj.items():
        pares.sort()
        i = 0
        while i + k - 1 < len(pares):
            v = pares[i:i + k]
            if v[-1][0] - v[0][0] <= span_max - 1:
                wins.append([j for _, j in v]); i += k       # sin solapamiento
            else:
                i += 1
    return wins

wins = ventanas_con_huecos(ids_c)
cob_est = sum(len(w) for w in wins) / len(ids_c)
print(f'Ventanas k={K}, span<={SPAN_MAX}: {len(wins)} · cobertura estructural {cob_est:.1%}')
print(f'(experimento 1, sin tolerancia a huecos: 42,9 %)')

# --- Asserts estructurales ---
for w in wins:
    ss = {parse(ids_c[i])[0] for i in w}
    assert len(ss) == 1, f'Ventana cruza sujeto: {[ids_c[i] for i in w]}'
    ts = sorted(parse(ids_c[i])[1] for i in w)
    assert ts[-1] - ts[0] <= SPAN_MAX - 1, 'Span excedido'
    assert len(set(ts)) == K, 'Sufijos repetidos'
print('✅ Asserts estructurales OK: ninguna ventana cruza sujeto ni excede el span.')

# Comparación con la regla vieja (span = k, o sea sin huecos)
wins_viejo = ventanas_con_huecos(ids_c, span_max=K)
print(f'\nRegla vieja (span<={K}): {len(wins_viejo)} ventanas, '
      f'cobertura {sum(len(w) for w in wins_viejo)/len(ids_c):.1%}')

# Distribución de faltantes por ventana
faltantes = collections.Counter(
    (max(parse(ids_c[i])[1] for i in w) - min(parse(ids_c[i])[1] for i in w) + 1) - K
    for w in wins)
print('Clips faltantes por ventana:', dict(sorted(faltantes.items())))

## 2. Agregación y métricas

In [ ]:
def agregar(wins, y, p, ids):
    y_w, p_w, part_w = [], [], []
    for w in wins:
        yy = y[w]
        y_w.append(int(yy.sum() * 2 > len(w)))     # mayoría, k impar
        p_w.append(float(p[w].mean()))             # media de probabilidades
        part_w.append(parse(ids[w[0]])[0])
    return np.array(y_w), np.array(p_w), np.array(part_w)

y_w, p_w, part_w = agregar(wins, y_c, p_c, ids_c)
print(f'Ventanas: {len(y_w)} · {len(set(part_w))} participantes · '
      f'trivial ventana {max((y_w==0).mean(), (y_w==1).mean()):.4f}')

def curva(p, y, coberturas=COBERTURAS, umbral=UMBRAL):
    conf = np.abs(p - umbral); orden = np.argsort(-conf)
    filas = []
    for c in coberturas:
        k = max(1, int(round(c * len(p))))
        sel = orden[:k]; ys = y[sel]; pred = (p[sel] > umbral).astype(int)
        triv = max((ys == 0).mean(), (ys == 1).mean()); acc = accuracy_score(ys, pred)
        filas.append({'cobertura': float(c), 'n': int(k),
                      'accuracy': round(float(acc), 4),
                      'trivial': round(float(triv), 4),
                      'brecha': round(float(acc - triv), 4),
                      'balanced_acc': round(float(balanced_accuracy_score(ys, pred)), 4),
                      'f1_macro': round(float(f1_score(ys, pred, average='macro')), 4)})
    return pd.DataFrame(filas)

df_w = curva(p_w, y_w)
df_c = curva(p_c, y_c)
print('\n=== POR VENTANA ===');  print(df_w.to_string(index=False))
print('\n=== POR CLIP (referencia, §12.1) ==='); print(df_c.to_string(index=False))
df_w.to_csv(f'{OUT_36}/curva_ventana_k5_span7.csv', index=False)

## 3. Bootstrap clusterizado por participante

La unidad de remuestreo sigue siendo el **participante**, no la ventana: las ventanas de un
mismo sujeto están correlacionadas.

In [ ]:
def boot(p, y, part, c=1.0, n_boot=N_BOOT, seed=SEED, umbral=UMBRAL):
    rng = np.random.default_rng(seed); parts = np.unique(part)
    idx = {q: np.where(part == q)[0] for q in parts}
    conf = np.abs(p - umbral)
    accs, brs, bas = (np.empty(n_boot) for _ in range(3))
    for i in range(n_boot):
        sel = rng.choice(parts, size=len(parts), replace=True)
        j = np.concatenate([idx[q] for q in sel])
        j = j[np.argsort(-conf[j])[:max(1, int(round(c * len(j))))]]
        ys = y[j]; ok = ((p[j] > umbral).astype(int) == ys)
        n0, n1 = (ys == 0).sum(), (ys == 1).sum()
        a = ok.mean(); t = max((ys == 0).mean(), (ys == 1).mean())
        accs[i] = a; brs[i] = a - t
        bas[i] = np.nan if (n0 == 0 or n1 == 0) else 0.5*(ok[ys==0].sum()/n0 + ok[ys==1].sum()/n1)
    pct = lambda v: [round(float(x), 4) for x in np.percentile(v[~np.isnan(v)], [2.5, 97.5])]
    return {'accuracy_IC95': pct(accs), 'brecha_IC95': pct(brs), 'balacc_IC95': pct(bas)}

for c in (1.00, 0.80, 0.70, 0.60):
    r = boot(p_w, y_w, part_w, c)
    fila = df_w[df_w.cobertura == c].iloc[0]
    print(f'c={c:.2f} | acc {fila.accuracy:.4f} {r["accuracy_IC95"]} | '
          f'BalAcc {fila.balanced_acc:.4f} {r["balacc_IC95"]} | brecha {fila.brecha:+.4f} '
          f'{r["brecha_IC95"]}')

## 4. Control obligatorio — ¿gana la ventana por agregar o por tener menos unidades?

Se compara contra **clips remuestreados al mismo n**: si la ganancia se explica solo por tener
menos unidades de evaluación, no hay efecto de agregación.

In [ ]:
rng = np.random.default_rng(SEED)
accs = []
for _ in range(200):
    sel = rng.choice(len(y_c), size=len(y_w), replace=False)
    accs.append(accuracy_score(y_c[sel], (p_c[sel] > UMBRAL).astype(int)))
print(f'Clips remuestreados a n={len(y_w)}: accuracy {np.mean(accs):.4f} ± {np.std(accs):.4f}')
print(f'Ventanas (cobertura 1,00):        accuracy {df_w.iloc[0].accuracy:.4f}')
print('Si son iguales, la ventana no agrega nada y el efecto es tamaño de muestra.')

## 5. Comparación pareada ventana vs. clip

No es pareada a nivel unidad (son unidades distintas), pero sí a nivel **participante**: en
cada remuestra se computan ambas sobre los mismos sujetos.

In [ ]:
def boot_ventana_vs_clip(n_boot=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed)
    parts = np.unique(part_c)
    iw = {q: np.where(part_w == q)[0] for q in parts}
    ic = {q: np.where(part_c == q)[0] for q in parts}
    d = np.empty(n_boot)
    okw = (p_w > UMBRAL).astype(int) == y_w
    okc = (p_c > UMBRAL).astype(int) == y_c
    def ba(y, ok):
        n0, n1 = (y == 0).sum(), (y == 1).sum()
        return np.nan if (n0 == 0 or n1 == 0) else 0.5*(ok[y==0].sum()/n0 + ok[y==1].sum()/n1)
    for i in range(n_boot):
        sel = rng.choice(parts, size=len(parts), replace=True)
        jw = np.concatenate([iw[q] for q in sel if len(iw[q])])
        jc = np.concatenate([ic[q] for q in sel])
        d[i] = ba(y_w[jw], okw[jw]) - ba(y_c[jc], okc[jc])
    d = d[~np.isnan(d)]
    lo, hi = np.percentile(d, [2.5, 97.5])
    return {'delta_balacc': round(float(d.mean()), 4),
            'IC95': [round(float(lo), 4), round(float(hi), 4)],
            'cruza_cero': bool(lo <= 0 <= hi)}

r_vc = boot_ventana_vs_clip()
print('BalAcc ventana − BalAcc clip (cobertura 1,00):', r_vc)
if r_vc['cruza_cero']:
    print('⚠️  El IC cruza cero: la ganancia por agregación NO se afirma (regla 6).')

## 6. Gráfico y veredicto

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for d_, lab in ((df_w, 'ventana k=5'), (df_c, 'clip')):
    ax[0].plot(d_.cobertura, d_.accuracy, 'o-', label=f'accuracy {lab}')
    ax[0].plot(d_.cobertura, d_.trivial, 's--', alpha=.5, label=f'trivial {lab}')
ax[0].axhline(ACC_OBJETIVO, color='r', ls=':', label='objetivo 0,70')
ax[0].axvline(COB_MIN, color='gray', ls=':')
ax[0].invert_xaxis(); ax[0].legend(fontsize=8); ax[0].set_xlabel('cobertura')
ax[1].plot(df_w.cobertura, df_w.balanced_acc, 'o-', label='BalAcc ventana')
ax[1].plot(df_c.cobertura, df_c.balanced_acc, 'o-', label='BalAcc clip')
ax[1].axhline(BALACC_PISO, color='r', ls=':', label=f'piso {BALACC_PISO}')
ax[1].axvline(COB_MIN, color='gray', ls=':')
ax[1].invert_xaxis(); ax[1].legend(fontsize=8); ax[1].set_xlabel('cobertura')
plt.tight_layout(); plt.savefig(f'{OUT_36}/ventana_vs_clip.png', dpi=120); plt.show()

print('================ VEREDICTO PASO 2 ================')
cands = []
for _, f in df_w[df_w.cobertura >= COB_MIN].iterrows():
    if f.accuracy >= ACC_OBJETIVO:
        r = boot(p_w, y_w, part_w, f.cobertura)
        if r['balacc_IC95'][0] >= BALACC_PISO:
            cands.append({'cobertura': f.cobertura, 'accuracy': f.accuracy,
                          'trivial': f.trivial, 'brecha': f.brecha,
                          'balanced_acc': f.balanced_acc, **r})
if not cands:
    mejor = df_w[df_w.cobertura >= COB_MIN].sort_values('accuracy').iloc[-1]
    print(f'❌ No se alcanza accuracy ≥ {ACC_OBJETIVO} con cobertura ≥ {COB_MIN}.')
    print(f'   Mejor admisible: accuracy {mejor.accuracy:.4f} a cobertura {mejor.cobertura:.2f} '
          f'(BalAcc {mejor.balanced_acc:.4f}, brecha {mejor.brecha:+.4f}).')
    print('   Documentar y pasar al paso 3 (backbone de afecto).')
else:
    elegida = sorted(cands, key=lambda r: -r['cobertura'])[0]
    print('✅ Configuración que cumple (se elige la MAYOR cobertura, no la mejor):')
    print(json.dumps(elegida, indent=2, ensure_ascii=False))
    print('\n⚠️  Cobertura y regla de ventana quedaron fijadas sobre OOF. El número no sesgado '
          'es el del cierre sobre Test.')

json.dump({'k': K, 'span_max': SPAN_MAX, 'cobertura_estructural': cob_est,
           'curva_ventana': df_w.to_dict('records'),
           'ventana_vs_clip': r_vc, 'candidatas': cands},
          open(f'{OUT_36}/paso2_agregacion.json', 'w'), indent=2, ensure_ascii=False, default=str)
print(f'\nGuardado. Evaluaciones sobre Test consumidas: 0.')